In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path('.').resolve()

MODEL_CFG = {
    'logistic': {
        'pre_glob': REPO_ROOT / 'reports' / '06_modelo_logistic' / '**' / 'resumen_modelos_logistic_regression.csv',
        'post_globs': [],  # no hay notebook de tuning dedicado para logistic en esta etapa
        'post_report_name': None,
    },
    'xgboost': {
        'pre_glob': REPO_ROOT / 'reports' / '08_modelo_xgboost' / '**' / 'resumen_modelos_xgboost.csv',
        'post_globs': [
            REPO_ROOT / 'reports' / '13_tuning_xgboost' / '**' / 'resumen_tuning_xgboost_all_candidates.csv',
            REPO_ROOT / 'reports' / '13_tuning_xgboost' / '**' / 'resumen_tuning_xgboost.csv',
        ],
        'post_report_name': 'classification_report_test_xgboost.csv',
    },
    'lightgbm': {
        'pre_glob': REPO_ROOT / 'reports' / '09_modelo_lightgbm' / '**' / 'resumen_modelos_lightgbm.csv',
        'post_globs': [
            REPO_ROOT / 'reports' / '11_tuning_lightgbm' / '**' / 'resumen_tuning_lightgbm_all_candidates.csv',
            REPO_ROOT / 'reports' / '11_tuning_lightgbm' / '**' / 'resumen_tuning_lightgbm.csv',
        ],
        'post_report_name': 'classification_report_test_lightgbm.csv',
    },
    'redmlp': {
        'pre_glob': REPO_ROOT / 'reports' / '10_modelo_redmlp' / '**' / 'resumen_modelos_redmlp.csv',
        'post_globs': [
            REPO_ROOT / 'reports' / '12_tuning_redmlp' / '**' / 'resumen_tuning_redmlp_all_candidates.csv',
            REPO_ROOT / 'reports' / '12_tuning_redmlp' / '**' / 'resumen_tuning_redmlp.csv',
        ],
        'post_report_name': 'classification_report_test_redmlp.csv',
    },
}


def latest_file(pattern: Path):
    files = sorted(REPO_ROOT.glob(str(pattern.relative_to(REPO_ROOT))), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


def latest_file_from_list(patterns):
    found = [latest_file(pat) for pat in patterns]
    found = [f for f in found if f is not None]
    if not found:
        return None
    return sorted(found, key=lambda p: p.stat().st_mtime)[-1]


def select_best_pre_candidates(df):
    req = ['cv_recall_target', 'cv_macro_f1']
    for c in req:
        if c not in df.columns:
            raise ValueError(f'Falta columna {c} en resumen pre-tuning')

    d = df.dropna(subset=req).copy()
    if d.empty:
        raise ValueError('Resumen pre-tuning sin filas validas')

    best_recall = d.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]
    best_general = d.sort_values(['cv_macro_f1', 'cv_recall_target'], ascending=False).iloc[0]

    return {
        'best_recall_target': best_recall,
        'best_general': best_general,
    }


def pick_pre_row(df, base_name, balance_name, fallback_row):
    d = df.copy()
    if 'balanceo' in d.columns:
        mask = (d['modelo'].astype(str) == str(base_name)) & (d['balanceo'].astype(str) == str(balance_name))
    else:
        mask = d['modelo'].astype(str) == str(base_name)
    if mask.any():
        return d.loc[mask].iloc[0]
    return fallback_row


def safe_get(row, key):
    try:
        return row.get(key)
    except Exception:
        return None


def read_class1_from_report(report_csv: Path):
    if report_csv is None or not report_csv.exists():
        return (None, None, None)
    try:
        rep = pd.read_csv(report_csv, index_col=0)
        cls = '1'
        if cls not in rep.index:
            return (None, None, None)
        f1 = rep.loc[cls, 'f1-score'] if 'f1-score' in rep.columns else None
        recall = rep.loc[cls, 'recall'] if 'recall' in rep.columns else None
        precision = rep.loc[cls, 'precision'] if 'precision' in rep.columns else None
        return (f1, recall, precision)
    except Exception:
        return (None, None, None)


In [2]:
rows = []

for model_name, cfg in MODEL_CFG.items():
    pre_file = latest_file(cfg['pre_glob'])
    post_file = latest_file_from_list(cfg.get('post_globs', []))

    if pre_file is None:
        print(f'[WARN] No se encontro resumen pre-tuning para {model_name}')
        continue

    pre_df = pd.read_csv(pre_file)
    pre_candidates = select_best_pre_candidates(pre_df)

    # Si no hay post-tuning, igual dejamos filas pre para ambos criterios.
    if post_file is None:
        for tag, pre_row in pre_candidates.items():
            row = {
                'modelo': model_name,
                'candidate_tag': tag,
                'pre_file': str(pre_file),
                'post_file': None,
                'pre_base_name': safe_get(pre_row, 'modelo'),
                'pre_balanceo': safe_get(pre_row, 'balanceo'),
                'post_base_name': None,
                'post_balanceo': None,
                'cv_recall_target_pre': safe_get(pre_row, 'cv_recall_target'),
                'cv_macro_f1_pre': safe_get(pre_row, 'cv_macro_f1'),
                'test_accuracy_pre': safe_get(pre_row, 'accuracy_test'),
                'test_f1_macro_pre': safe_get(pre_row, 'macro_f1_test'),
                'f1_1_test_pre': safe_get(pre_row, 'f1_1_test'),
                'recall_1_test_pre': safe_get(pre_row, 'recall_1_test'),
                'precision_1_test_pre': safe_get(pre_row, 'precision_1_test'),
                'cv_recall_target_post': None,
                'cv_macro_f1_post': None,
                'test_accuracy_post': None,
                'test_f1_macro_post': None,
                'f1_1_test_post': None,
                'recall_1_test_post': None,
                'precision_1_test_post': None,
            }
            for base_metric in ['cv_recall_target', 'cv_macro_f1', 'test_accuracy', 'test_f1_macro', 'f1_1_test', 'recall_1_test', 'precision_1_test']:
                row[f'delta_{base_metric}'] = None
            rows.append(row)
        print(f'[INFO] {model_name}: sin post-tuning, se reporta solo pre-tuning')
        continue

    post_df = pd.read_csv(post_file)

    # Formato nuevo: resumen_tuning_*_all_candidates.csv
    if 'candidate_tag' in post_df.columns:
        iter_rows = [r for _, r in post_df.iterrows()]
    else:
        # Formato anterior (una sola fila)
        iter_rows = [post_df.iloc[0]] if not post_df.empty else []

    if not iter_rows:
        print(f'[WARN] {model_name}: post-tuning vacio en {post_file}')
        continue

    for post_row in iter_rows:
        tag = post_row.get('candidate_tag') or 'best_recall_target'
        base_name = post_row.get('base_name')
        balance_name = post_row.get('balanceo')

        fallback_pre = pre_candidates['best_general'] if tag == 'best_general' else pre_candidates['best_recall_target']
        pre_row = pick_pre_row(pre_df, base_name, balance_name, fallback_pre)

        report_name = cfg.get('post_report_name')
        report_csv = None
        if report_name:
            if 'candidate_tag' in post_df.columns and pd.notna(tag):
                report_csv = post_file.parent / str(tag) / report_name
            else:
                report_csv = post_file.parent / report_name
        f1_1_post, recall_1_post, precision_1_post = read_class1_from_report(report_csv)

        row = {
            'modelo': model_name,
            'candidate_tag': tag,
            'pre_file': str(pre_file),
            'post_file': str(post_file),
            'pre_base_name': safe_get(pre_row, 'modelo'),
            'pre_balanceo': safe_get(pre_row, 'balanceo'),
            'post_base_name': base_name,
            'post_balanceo': balance_name,
            'cv_recall_target_pre': safe_get(pre_row, 'cv_recall_target'),
            'cv_macro_f1_pre': safe_get(pre_row, 'cv_macro_f1'),
            'test_accuracy_pre': safe_get(pre_row, 'accuracy_test'),
            'test_f1_macro_pre': safe_get(pre_row, 'macro_f1_test'),
            'f1_1_test_pre': safe_get(pre_row, 'f1_1_test'),
            'recall_1_test_pre': safe_get(pre_row, 'recall_1_test'),
            'precision_1_test_pre': safe_get(pre_row, 'precision_1_test'),
            'cv_recall_target_post': post_row.get('cv_best_recall_target'),
            'cv_macro_f1_post': post_row.get('cv_best_f1_macro'),
            'test_accuracy_post': post_row.get('test_accuracy'),
            'test_f1_macro_post': post_row.get('test_f1_macro'),
            'f1_1_test_post': f1_1_post,
            'recall_1_test_post': recall_1_post,
            'precision_1_test_post': precision_1_post,
        }

        for base_metric in ['cv_recall_target', 'cv_macro_f1', 'test_accuracy', 'test_f1_macro', 'f1_1_test', 'recall_1_test', 'precision_1_test']:
            pre_v = row.get(f'{base_metric}_pre')
            post_v = row.get(f'{base_metric}_post')
            row[f'delta_{base_metric}'] = (post_v - pre_v) if pd.notna(pre_v) and pd.notna(post_v) else None

        rows.append(row)

df_cmp = pd.DataFrame(rows)
df_cmp = df_cmp.sort_values(['modelo', 'candidate_tag']).reset_index(drop=True)
df_cmp


[INFO] logistic: sin post-tuning, se reporta solo pre-tuning


,modelo,candidate_tag,pre_file,post_file,pre_base_name,pre_balanceo,post_base_name,post_balanceo,cv_recall_target_pre,cv_macro_f1_pre,...,f1_1_test_post,recall_1_test_post,precision_1_test_post,delta_cv_recall_target,delta_cv_macro_f1,delta_test_accuracy,delta_test_f1_macro,delta_f1_1_test,delta_recall_1_test,delta_precision_1_test
0,lightgbm,best_recall_target,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,MinMax_ORIGINAL_PCA30,SMOTEENN,MinMax_ORIGINAL_PCA30,SMOTEENN,0.901957,0.252281,...,NaN,NaN,NaN,-0.016794,0.004002,None,None,None,None,None
1,logistic,best_general,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,None,Standard_FE_ALL,UNDER,None,None,0.352417,0.435425,...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None
2,logistic,best_recall_target,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,None,NoNorm_FE_ALL,SMOTEENN,None,None,0.976049,0.075500,...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None
3,redmlp,best_recall_target,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,NoNorm_ORIGINAL_MI,SMOTEENN,NoNorm_ORIGINAL_MI,SMOTEENN,0.923715,0.197538,...,NaN,NaN,NaN,0.009807,-0.012019,None,None,None,None,None
4,xgboost,best_recall_target,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,C:\Users\Gonzalo\Documents\GitHub\TesisAustral...,Robust_ORIGINAL_ALL,SMOTEENN,Robust_ORIGINAL_ALL,SMOTEENN,0.900896,0.266330,...,0.207,0.950752,0.116144,0.046022,-0.064458,None,None,None,None,None


In [3]:
out_dir = REPO_ROOT / 'reports' / '14_comparacion_tuning' / pd.Timestamp.today().strftime('%Y%m%d')
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / 'comparacion_pre_vs_post_tuning_all_models.csv'
df_cmp.to_csv(out_file, index=False)

print(f'Comparacion guardada en: {out_file}')
print('Filas:', len(df_cmp))


Comparacion guardada en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\14_comparacion_tuning\20260308\comparacion_pre_vs_post_tuning_all_models.csv
Filas: 5
